# Deteksi Kepatuhan dan Pelanggaran APD - Pipeline Colab

Menjalankan seluruh eksperimen: unduh data, latih tiga model, evaluasi, uji statistik, dan buat gambar.

**WAJIB sebelum mulai:** `Runtime -> Change runtime type -> T4 GPU`, lalu Save.

Perkiraan total 6-9 jam. Semua hasil tersimpan di Google Drive dan setiap pelatihan
dapat dilanjutkan, jadi sesi yang terputus tidak menghapus progres - cukup jalankan
ulang sel 1-6 lalu sel pelatihan.

## 1. Pastikan GPU aktif

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        'GPU TIDAK AKTIF.\n'
        'Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save,\n'
        'lalu jalankan ulang sel ini.'
    )

print('GPU   :', torch.cuda.get_device_name(0))
print('VRAM  : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
print('torch :', torch.__version__)

## 2. Sambungkan Google Drive

Kode dan hasil disimpan di Drive supaya bertahan antar-sesi. Dataset disimpan di
disk lokal Colab karena membaca ribuan citra dari Drive jauh lebih lambat.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
PROJECT = pathlib.Path('/content/drive/MyDrive/ppe-research')
PROJECT.mkdir(parents=True, exist_ok=True)
print('Folder proyek:', PROJECT)

## 3. Pastikan kode proyek ada

Unggah folder `configs/` dan `scripts/` ke `MyDrive/ppe-research/` lewat antarmuka
Google Drive, lalu jalankan sel ini untuk memastikan semuanya lengkap.

In [ ]:
%cd /content/drive/MyDrive/ppe-research

wajib = [
    'configs/experiments.yaml',
    'scripts/common.py',
    'scripts/detection_eval.py',
    'scripts/test_detection_eval.py',
    'scripts/01_download_dataset.py',
    'scripts/02_prepare_dataset.py',
    'scripts/03_train.py',
    'scripts/04_evaluate.py',
    'scripts/05_statistics.py',
    'scripts/06_figures.py',
]
kurang = [p for p in wajib if not os.path.exists(p)]
if kurang:
    raise SystemExit('Berkas belum lengkap:\n  ' + '\n  '.join(kurang))
print('Kode proyek lengkap -', len(wajib), 'berkas.')

## 4. Pasang dependensi

In [ ]:
!pip install -q -U ultralytics kaggle

import ultralytics
ultralytics.checks()

## 5. Kredensial Kaggle

Ambil di kaggle.com -> Settings -> bagian API -> **Create New Token**.
Token berupa teks berawalan `KGAT_`. Tempelkan saat diminta di bawah;
masukannya tersembunyi dan hanya tersimpan di sesi Colab ini.

In [ ]:
import getpass, pathlib

tok_file = pathlib.Path.home() / '.kaggle' / 'access_token'
if tok_file.is_file() and tok_file.stat().st_size > 0:
    print('Token Kaggle sudah ada.')
else:
    token = getpass.getpass('Tempel token Kaggle (KGAT_...): ').strip()
    if not token:
        raise SystemExit('Token kosong.')
    tok_file.parent.mkdir(parents=True, exist_ok=True)
    tok_file.write_text(token)
    tok_file.chmod(0o600)
    print('Token tersimpan.')

from kaggle.api.kaggle_api_extended import KaggleApi
KaggleApi().authenticate()
print('Autentikasi Kaggle berhasil.')

## 6. Unduh dan siapkan dataset

Sekitar 1,85 GB, biasanya 2-5 menit pada koneksi Colab. Dataset sudah berformat
YOLO dengan pembagian train/val/test terstratifikasi, jadi tidak perlu diproses ulang.

In [ ]:
!python scripts/01_download_dataset.py --dest /content/data/PPE
!python scripts/02_prepare_dataset.py --root /content/data/PPE
!cat data/ppe.yaml

## 7. Verifikasi modul evaluasi

Seluruh angka pada naskah bergantung pada modul ini, jadi kebenarannya diuji dulu
terhadap kasus yang jawabannya dapat dihitung tangan. **Harus 21 lulus, 0 gagal.**

In [ ]:
!python scripts/test_detection_eval.py

## 8. Pelatihan tiga model

`yolov8n`, `yolo11n`, `yolo26n` - masing-masing 60 epoch, hiperparameter identik.
Perkiraan 2-3 jam per model pada T4.

Bila sesi terputus: jalankan ulang sel 1-6, lalu sel ini. Model yang sudah selesai
dilewati otomatis, yang terpotong dilanjutkan dari checkpoint terakhir.

Agar sesi tidak terputus karena dianggap menganggur, biarkan tab Colab terbuka.

In [ ]:
!python scripts/03_train.py --all --resume

## 9. Evaluasi pada himpunan uji

Inilah satu-satunya tahap yang menyentuh split uji.

In [ ]:
!python scripts/04_evaluate.py --all

## 10. Uji signifikansi (bootstrap berpasangan)

Berjalan di CPU, sekitar 20-40 menit untuk 1.000 resample.
Tambahkan `--fast` bila ingin uji coba cepat (hanya IoU 0,50).

In [ ]:
!python scripts/05_statistics.py --n-bootstrap 1000

## 11. Gambar dan tabel untuk naskah

In [ ]:
!python scripts/06_figures.py

from IPython.display import Markdown, display
display(Markdown(open('results/analysis/tabel_naskah.md', encoding='utf-8').read()))

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('results/figures/*.png')):
    print(p)
    display(Image(p))

## 12. Unduh hasil

Mengemas gambar, tabel, dan JSON metrik. Bobot model tetap di Drive karena besar.

Setelah arsip terunduh, buka Claude Code lagi, letakkan isinya di folder `results/`
proyek, lalu minta lanjutkan analisis dan pengisian naskah.

In [ ]:
!cd /content/drive/MyDrive/ppe-research && zip -r /content/hasil_ppe.zip results/figures results/analysis results/runs/*/results.csv results/runs/*/meta.json

from google.colab import files
files.download('/content/hasil_ppe.zip')